# Gatekeeper v6 - Stage 1 (medical / non-medical), bone_fracture removed

**What changed vs v5:** `bone_fracture` is no longer a positive source (6 modalities now). Nothing else is intentionally different:
full retrain from ImageNet weights (not a fine-tune), EfficientNet-B0 + wide head, **plain unweighted CrossEntropyLoss**, resolution-degradation aug at p=0.3,
224x224, lr 1e-3 / wd 1e-4, ReduceLROnPlateau, per-epoch best-checkpoint + resume.

**Needs:** `scripts/router_common.py` next to the other scripts. **Do not trust the in-distribution number alone** - Steps 7-8 (per-source + real-image PASS/FAIL) are the real test.

## 1. Setup

In [1]:
import sys, time, shutil, importlib
from pathlib import Path
ROOT = Path(r"E:\programming\CliniqAI")
sys.path.insert(0, str(ROOT / "scripts"))
import torch
import router_common as rc
importlib.reload(rc)            # picks up edits to router_common.py without a kernel restart
torch.backends.cudnn.benchmark = True
rc.set_seed(42)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch", torch.__version__, "| device:", DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "")
DATA_RAW   = ROOT / "data" / "raw"
MODELS_DIR = ROOT / "models"

torch 2.5.0+cu124 | device: cuda NVIDIA GeForce RTX 4050 Laptop GPU


## 2. Config

In [2]:
TAG          = "gatekeeper_v6_no_bone"
FINAL_PATH   = MODELS_DIR / f"{TAG}.pt"

CLASS_NAMES  = ["non_medical", "medical"]
# Set this to your CURRENT gatekeeper checkpoint (v4/v5 .pt) so class order is copied from it,
# in case the backend relies on the index of "medical". None = use the order above.
REFERENCE_CKPT = None

NEGATIVE_SOURCES   = None   # None = auto-discover folders in data/raw (medical-looking names are rejected). Or give an explicit list.
EXTRA_EXCLUDE_DIRS = ["breast_histology", "cifar10"]   # old IDC patches; deprecated v1 negative (v5 did not use CIFAR-10)
HOLDOUT_SUBSTR     = ("fruits",)   # held out of training, used as a generalisation test (Step 8)

BATCH_SIZE = 64      # lower to 32 if you hit CUDA OOM
NUM_WORKERS = 4      # keep at 4 (higher counts crashed/hung before)
EPOCHS, PATIENCE = 30, 7
LR, WD = 1e-3, 1e-4
DEGRADE_P = 0.3
MAX_NEG_PER_POS = None   # None = use all negatives like v5. Removing bone_fracture raises neg:pos; see note in Step 3.
RESUME = True            # picks up <TAG>.checkpoint_inprogress.pt if it exists

if REFERENCE_CKPT:
    CLASS_NAMES = list(torch.load(REFERENCE_CKPT, map_location="cpu", weights_only=False)["class_names"])
    print("Class order copied from reference checkpoint:", CLASS_NAMES)
assert sorted(CLASS_NAMES) == ["medical", "non_medical"]
print("CLASS_NAMES =", CLASS_NAMES)

CLASS_NAMES = ['non_medical', 'medical']


## 3. Build sample lists + group-aware splits

In [3]:
splits, info = rc.build_gatekeeper_samples(
    DATA_RAW, CLASS_NAMES, neg_sources=NEGATIVE_SOURCES, extra_exclude=EXTRA_EXCLUDE_DIRS,
    holdout=HOLDOUT_SUBSTR, max_neg_per_pos=MAX_NEG_PER_POS)
rc.print_split_table(splits, CLASS_NAMES)
rc.assert_no_group_overlap(splits)

print(f"\nPositives: {info['n_pos_total']}   Negatives: {info['n_neg_total']}   neg:pos = {info['n_neg_total']/info['n_pos_total']:.2f}x")
print("v5 reference: 191,020 negatives / 58,103 positives (that included bone_fracture, up to 12,000 images).")
if info["n_neg_total"] != 191020:
    print("NOTE: negative count differs from v5's 191,020 -> check the source table above for a folder that was added/missed/duplicated (e.g. the second food101 download).")
print("NOTE: with bone_fracture gone the neg:pos ratio is higher than in v5. v5's open problem was FALSE NEGATIVES on real chest x-rays/retinal scans, "
      "so if Step 7/8 show that again, set MAX_NEG_PER_POS = 2.0 and rerun - but change that ONE thing only.")

Positives (6 modalities, bone_fracture NOT included):
  brain_mri          brain_tumor            found   7200  using   7200
  breast_histology   breast_histology_v2    found   7909  using   7909
  kidney_ct          kidney_stone           found  12446  using  12000
  pneumonia          pneumonia              found   4751  using   4751
  retinal            diabetic_retinopathy   found   3662  using   3662
  skin_lesion        skin_lesion            found  25331  using  12000

Auto-discovered negative sources (1); skipped dirs: ['brain_tumor', 'breast_histology', 'breast_histology_v2', 'cifar10', 'diabetic_retinopathy', 'kidney_stone', 'manual_sanity_check', 'manual_sanity_check_negative', 'pneumonia', 'skin_lesion']
Negatives:
  gatekeeper_negatives_v2       526476

source                       class           train      val     test
brain_mri                    medical          5760      720      720
breast_histology             medical          6356      821      732
gatekeeper_negat

## 4. Datasets, loaders, model

In [4]:
train_ds = rc.ListDataset(splits["train"], rc.train_transform(degrade_p=DEGRADE_P))
val_ds   = rc.ListDataset(splits["val"],   rc.eval_transform())
test_ds  = rc.ListDataset(splits["test"],  rc.eval_transform())
train_loader = rc.make_loader(train_ds, BATCH_SIZE, shuffle=True,  num_workers=NUM_WORKERS)
val_loader   = rc.make_loader(val_ds,   BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, persistent=True)
model = rc.build_model(len(CLASS_NAMES), pretrained=True)   # full retrain from ImageNet weights
print(len(train_ds), len(val_ds), len(test_ds), "| params:", sum(p.numel() for p in model.parameters()) / 1e6, "M")

459229 57430 57339 | params: 4.664446 M


## 5. Train  (safe to interrupt - re-running this cell resumes from the best checkpoint; a kernel restart means re-running Steps 1-4 first)

In [5]:
history = rc.train_model(model, train_loader, val_loader, CLASS_NAMES, MODELS_DIR, TAG, DEVICE,
                         epochs=EPOCHS, lr=LR, weight_decay=WD, patience=PATIENCE, resume=RESUME)

epoch 1/30:   0%|          | 0/7175 [00:12<?, ?it/s]

epoch  1/30 | train 0.0115/0.9965 | val 0.0019/0.9994 | lr 1.0e-03 | 30.1 min  * best


epoch 2/30:   0%|          | 0/7175 [00:00<?, ?it/s]

epoch  2/30 | train 0.0060/0.9983 | val 0.0010/0.9997 | lr 1.0e-03 | 26.1 min  * best


epoch 3/30:   0%|          | 0/7175 [00:00<?, ?it/s]

...

epoch 27/30 | train 0.0002/1.0000 | val 0.0001/1.0000 | lr 6.3e-05 | 27.0 min


epoch 28/30:   0%|          | 0/7175 [00:00<?, ?it/s]

epoch 28/30 | train 0.0002/0.9999 | val 0.0002/1.0000 | lr 3.1e-05 | 29.6 min


epoch 29/30:   0%|          | 0/7175 [00:00<?, ?it/s]

epoch 29/30 | train 0.0001/1.0000 | val 0.0001/1.0000 | lr 3.1e-05 | 27.7 min
Early stopping: no val_loss improvement for 7 epochs.
Loaded best weights (val_loss 0.0000, epoch 22)


## 6. Save (before evaluating, so nothing is lost if an eval cell errors)

In [6]:
rc.save_final(model, CLASS_NAMES, history, FINAL_PATH, extra={"excluded_modalities": ["bone_fracture"], "trained_modalities": rc.MODALITIES, "version": "v6"})
COPY_TO_BACKEND = False   # you have been bitten by root-models/ vs backend/models/ drift before - copy deliberately
if COPY_TO_BACKEND:
    (ROOT / "backend" / "models").mkdir(exist_ok=True)
    shutil.copy2(FINAL_PATH, ROOT / "backend" / "models" / FINAL_PATH.name)

Saved E:\programming\CliniqAI\models\gatekeeper_v6_no_bone.pt  +  gatekeeper_v6_no_bone.history.json   (round-trip load OK, classes=['non_medical', 'medical'])


## 7. Test-set breakdown (per class + per SOURCE - don't read the aggregate on its own)

In [7]:
probs = rc.predict_probs(model, test_ds, DEVICE, num_workers=NUM_WORKERS)
cm = rc.report(probs, test_ds.samples, CLASS_NAMES, "GATEKEEPER v6 TEST")

predict:   0%|          | 0/896 [00:17<?, ?it/s]


===== GATEKEEPER v6 TEST: overall acc 1.0000  (n=57339) =====
Per-class recall:
  non_medical        1.0000   (n=52647)
  medical            0.9996   (n=4692)
Confusion matrix (rows = true, cols = predicted):
             non_medic   medical
 non_medical     52647         0
     medical         2      4690
Per-source accuracy (worst first):
  skin_lesion                  0.9983   (n=1200)
  gatekeeper_negatives_v2      1.0000   (n=52647)
  kidney_ct                    1.0000   (n=1200)
  pneumonia                    1.0000   (n=474)
  brain_mri                    1.0000   (n=720)
  breast_histology             1.0000   (n=732)
  retinal                      1.0000   (n=366)


## 8. Real-world checks (these decide whether v6 gets deployed)
Folders (put real images in them; each missing/empty folder is flagged as "proves nothing"):
- `data/raw/manual_sanity_check_negative/` - real NON-medical images -> must be `non_medical`
- `data/raw/manual_sanity_check_medical/` - real medical images from your 6 modalities (esp. real chest x-rays + retinal scans, the v5 false-negative cases) -> must be `medical`
- `fruits` (held out of training) is checked automatically

In [8]:
res = {}
res["negative_real"] = rc.sanity_folder(model, DATA_RAW / "manual_sanity_check_negative", "non_medical", CLASS_NAMES, DEVICE)
res["medical_real"]  = rc.sanity_folder(model, DATA_RAW / "manual_sanity_check_medical",  "medical",     CLASS_NAMES, DEVICE)

# held-out negative source(s): none of these images were seen in training
import random
for h in HOLDOUT_SUBSTR:
    for parent in (DATA_RAW, DATA_RAW / "gatekeeper_negatives_v2"):
        for d in sorted(p for p in parent.iterdir() if p.is_dir() and h in p.name.lower()):
            files = rc.list_images(d)
            files = random.Random(0).sample(files, min(500, len(files)))
            prs = rc._predict_files(model, files, DEVICE)
            acc = sum(CLASS_NAMES[int(p.argmax())] == "non_medical" for p in prs) / len(prs)
            print(f"[holdout {d.name}] classified non_medical: {acc:.1%}  (n={len(files)})")
print("\nSummary:", res)

  [PASS] apple.jpg                                     -> non_medical      non_medical=1.000  medical=0.000
  [PASS] bird.jpg                                      -> non_medical      non_medical=1.000  medical=0.000
  [PASS] data_batch_1_1263.png                         -> non_medical      non_medical=1.000  medical=0.000
  [PASS] data_batch_1_292.png                          -> non_medical      non_medical=1.000  medical=0.000
  [PASS] data_batch_3_9536.png                         -> non_medical      non_medical=1.000  medical=0.000
[manual_sanity_check_negative] expected 'non_medical': 5/5 passed
[manual_sanity_check_medical] WARNING: E:\programming\CliniqAI\data\raw\manual_sanity_check_medical is missing/empty -> this check proves nothing yet
[holdout fruits] classified non_medical: 100.0%  (n=500)

Summary: {'negative_real': (5, 5), 'medical_real': None}


## 9. What happens to bone x-rays now? (information only)
`bone_fracture` is no longer a positive class and the gatekeeper never saw it as a negative either, so this shows what the system will do with a bone x-ray: rejected as non-medical, or passed on to Stage 2 (where it will be forced into one of the 6 modalities). Use this to decide how to handle it in the UI/backend.

In [9]:
rc.probe_folder(model, DATA_RAW / "bone_fracture", CLASS_NAMES, DEVICE, n=300, title="bone x-rays (unseen modality)")

[bone x-rays (unseen modality)] folder missing/empty - skipped


## 10. Deploy checklist
1. Step 7 + 8 all sane (no per-source collapse, real medical + real non-medical PASS).
2. Copy `models/gatekeeper_v6_no_bone.pt` to `backend/models/` if you did not set `COPY_TO_BACKEND`.
3. Point `ROUTER_STAGE1_PATH` at it. Commit this notebook (+ `scripts/router_common.py`) on `feature/router`.